In [0]:
%pip install --upgrade "databricks-sdk>=0.118.0" "psycopg[binary]>=3.1.0" -q
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Project, ProjectSpec

w = WorkspaceClient()

PROJECT_ID = "volta-ind"

op = w.postgres.create_project(
    project=Project(spec=ProjectSpec(display_name="Volta Industrial", pg_version=17)),
    project_id=PROJECT_ID,
)
project = op.wait()
print("Created:", project.name)

In [0]:
from databricks.sdk import WorkspaceClient

PROJECT_ID = "volta-ind"
BRANCH_ID = "production"
DATABASE = "databricks_postgres"

w = WorkspaceClient()

# Get the dev branch endpoint host
for e in w.postgres.list_endpoints(parent=f"projects/{PROJECT_ID}/branches/{BRANCH_ID}"):
    HOST = e.status.hosts.host
    print(f"Endpoint: {e.name}")
    print(f"Host: {HOST}")

In [0]:
# List branches
print("=== Branches ===")
for b in w.postgres.list_branches(parent=f"projects/{PROJECT_ID}"):
    print(f"  {b.name}  state={b.status.current_state}")

# List endpoints
print("\n=== Endpoints ===")
for e in w.postgres.list_endpoints(parent=f"projects/{PROJECT_ID}/branches/{BRANCH_ID}"):
    print(f"  {e.name}  host={e.status.hosts.host}")

# List databases
print("\n=== Databases ===")
for d in w.postgres.list_databases(parent=f"projects/{PROJECT_ID}/branches/{BRANCH_ID}"):
    print(f"  {d.name}")

# List schemas in databricks_postgres
print("\n=== Schemas ===")
print("  (check via SQL below)")

In [0]:
from databricks.sdk.service.postgres import Branch, BranchSpec

DEV_BRANCH_ID = "dev"

op = w.postgres.create_branch(
    parent=f"projects/{PROJECT_ID}",
    branch=Branch(spec=BranchSpec(
        source_branch=f"projects/{PROJECT_ID}/branches/{BRANCH_ID}",
        no_expiry=True,
    )),
    branch_id=DEV_BRANCH_ID,
)
dev_branch = op.wait()
print(f"Created: {dev_branch.name}  state={dev_branch.status.current_state}")

# Get the dev branch endpoint
for e in w.postgres.list_endpoints(parent=f"projects/{PROJECT_ID}/branches/{DEV_BRANCH_ID}"):
    DEV_HOST = e.status.hosts.host
    print(f"Dev endpoint: {e.name}")
    print(f"Dev host: {DEV_HOST}")

In [0]:
# --- Source Delta tables (UC) ---
CATALOG = "tech_summit_jinie_group_catalog"
SCHEMA = "default"

SYNC_MAP = {
    # target_table: (source_delta_table, pk_column, select_columns or None for all)
    "line_status": (
        f"{CATALOG}.{SCHEMA}.gold_line_status",
        "line_id",
        None,  # all columns
    ),
    "open_atrisk": (
        f"{CATALOG}.{SCHEMA}.gold_open_atrisk",
        "line_id",
        ["line_id", "failure_risk_score", "downtime_exposure_usd", "open_wo_count",
         "part_local", "candidate_part_id", "part_lead_time_days", "part_unit_cost_usd", "criticality"],
    ),
    "maintenance_recommendations": (
        f"{CATALOG}.{SCHEMA}.gold_maintenance_recommendations",
        "line_id",
        None,  # all columns
    ),
    "parts": (
        f"{CATALOG}.{SCHEMA}.raw_parts",
        "part_id",
        ["part_id", "part_name", "part_type", "machine_type", "unit_cost_usd",
         "lead_time_days", "local_stock_qty", "description", "is_active"],
    ),
}

print(f"Sync map configured: {list(SYNC_MAP.keys())}")

In [0]:
import psycopg
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# --- Lakebase connection ---
ENDPOINT_NAME = f"projects/{PROJECT_ID}/branches/dev/endpoints/primary"
ep = w.postgres.get_endpoint(name=ENDPOINT_NAME)
host = ep.status.hosts.host
user = w.current_user.me().user_name
token = w.postgres.generate_database_credential(endpoint=ENDPOINT_NAME).token

BATCH_SIZE = 2000
PG_SCHEMA = "app"

def sync_table(target_table, source_delta, pk_col, select_cols, conn):
    """Read a Delta table via Spark and batch-insert into Lakebase."""
    # Read from Delta
    df = spark.read.table(source_delta)
    if select_cols:
        df = df.select(*select_cols)
    pdf = df.toPandas()
    cols = list(pdf.columns)
    print(f"  {source_delta} → app.{target_table}: {len(pdf)} rows, {len(cols)} cols")

    if pdf.empty:
        print("    ⚠ No rows to sync")
        return 0

    # Build INSERT ... ON CONFLICT DO NOTHING
    col_list = ", ".join(cols)
    placeholders = ", ".join(["%s"] * len(cols))
    sql = f"""INSERT INTO {PG_SCHEMA}.{target_table} ({col_list})
             VALUES ({placeholders})
             ON CONFLICT ({pk_col}) DO NOTHING"""

    inserted = 0
    with conn.cursor() as cur:
        for start in range(0, len(pdf), BATCH_SIZE):
            batch = pdf.iloc[start : start + BATCH_SIZE]
            rows = []
            for _, row in batch.iterrows():
                vals = []
                for c in cols:
                    v = row[c]
                    # Convert NaN/NaT to None
                    if v is None or (isinstance(v, float) and str(v) == "nan"):
                        vals.append(None)
                    # action_ranking: stringify for JSONB
                    elif c == "action_ranking" and isinstance(v, str):
                        vals.append(v)  # already a JSON string
                    else:
                        vals.append(v)
                rows.append(tuple(vals))
            cur.executemany(sql, rows)
            inserted += cur.rowcount
        conn.commit()
    print(f"    ✓ {inserted} rows inserted (skipped duplicates)")
    return inserted

# --- Run sync ---
with psycopg.connect(host=host, dbname=DATABASE, user=user,
                     password=token, sslmode="require") as conn:
    total = 0
    for target, (source, pk, cols) in SYNC_MAP.items():
        total += sync_table(target, source, pk, cols, conn)
    print(f"\n✅ Sync complete — {total} total rows inserted")

In [0]:
# CDF is required for CONTINUOUS synced tables.
# gold_line_status and gold_open_atrisk are MATERIALIZED_VIEWs (no CDF support) → SNAPSHOT.
# gold_maintenance_recommendations and raw_parts are MANAGED → enable CDF → CONTINUOUS.

spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.gold_maintenance_recommendations SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.raw_parts SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print("CDF enabled on gold_maintenance_recommendations and raw_parts")

In [0]:
import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import (
    SyncedTable,
    SyncedTableSyncedTableSpec,
    SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy as SyncPolicy,
)

w = WorkspaceClient()
PROD_BRANCH_PATH = f"projects/{PROJECT_ID}/branches/production"
PROD_ENDPOINT = f"{PROD_BRANCH_PATH}/endpoints/primary"

# All 4 tables synced as one-time snapshot into app.* on production.
# Managed CONTINUOUS sync cannot use the original UC names (already taken by another project),
# and MVs don't support managed sync at all. Manual snapshot gives us the exact Postgres
# table names we need in the app schema.

print("=== One-time snapshot sync to production app schema ===")

ep = w.postgres.get_endpoint(name=PROD_ENDPOINT)
prod_host = ep.status.hosts.host
prod_user = w.current_user.me().user_name
prod_token = w.postgres.generate_database_credential(endpoint=PROD_ENDPOINT).token

TABLE_DDL = {
    "line_status": """CREATE TABLE IF NOT EXISTS app.line_status (
        line_id TEXT PRIMARY KEY, plant_id TEXT, line_name TEXT, machine_type TEXT,
        criticality TEXT, plant_lat DOUBLE PRECISION, plant_lng DOUBLE PRECISION,
        vibration_rms DOUBLE PRECISION, temperature_c DOUBLE PRECISION,
        utilization_pct DOUBLE PRECISION, failure_risk_score DOUBLE PRECISION,
        open_wo_count BIGINT, has_open_corrective BOOLEAN, part_local BOOLEAN,
        risk_signal_score NUMERIC(2,1), downtime_exposure_usd DOUBLE PRECISION, risk_band TEXT
    )""",
    "open_atrisk": """CREATE TABLE IF NOT EXISTS app.open_atrisk (
        line_id TEXT PRIMARY KEY, failure_risk_score DOUBLE PRECISION,
        downtime_exposure_usd DOUBLE PRECISION, open_wo_count BIGINT,
        part_local BOOLEAN, candidate_part_id TEXT, part_lead_time_days INTEGER,
        part_unit_cost_usd DOUBLE PRECISION, criticality TEXT
    )""",
    "maintenance_recommendations": """CREATE TABLE IF NOT EXISTS app.maintenance_recommendations (
        line_id TEXT PRIMARY KEY, recommended_action TEXT,
        predicted_downtime_cost_avoided_usd DOUBLE PRECISION,
        predicted_net_value_usd DOUBLE PRECISION,
        action_ranking JSONB, scored_at TIMESTAMP
    )""",
    "parts": """CREATE TABLE IF NOT EXISTS app.parts (
        part_id TEXT PRIMARY KEY, part_name TEXT, part_type TEXT, machine_type TEXT,
        unit_cost_usd DOUBLE PRECISION, lead_time_days INTEGER, local_stock_qty INTEGER,
        description TEXT, is_active BOOLEAN
    )""",
}

TABLE_SYNC = {
    "line_status": (f"{CATALOG}.{SCHEMA}.gold_line_status", "line_id", None),
    "open_atrisk": (f"{CATALOG}.{SCHEMA}.gold_open_atrisk", "line_id",
        ["line_id", "failure_risk_score", "downtime_exposure_usd", "open_wo_count",
         "part_local", "candidate_part_id", "part_lead_time_days", "part_unit_cost_usd", "criticality"]),
    "maintenance_recommendations": (f"{CATALOG}.{SCHEMA}.gold_maintenance_recommendations", "line_id", None),
    "parts": (f"{CATALOG}.{SCHEMA}.raw_parts", "part_id",
        ["part_id", "part_name", "part_type", "machine_type", "unit_cost_usd",
         "lead_time_days", "local_stock_qty", "description", "is_active"]),
}

BATCH_SIZE = 2000

with psycopg.connect(host=prod_host, dbname=DATABASE, user=prod_user,
                     password=prod_token, sslmode="require") as conn:
    with conn.cursor() as cur:
        for tbl, ddl in TABLE_DDL.items():
            cur.execute(ddl)
        conn.commit()

    for tbl, (source, pk, select_cols) in TABLE_SYNC.items():
        df = spark.read.table(source)
        if select_cols:
            df = df.select(*select_cols)
        pdf = df.toPandas()
        cols = list(pdf.columns)
        col_list = ", ".join(cols)
        placeholders = ", ".join(["%s"] * len(cols))
        sql = f"INSERT INTO app.{tbl} ({col_list}) VALUES ({placeholders}) ON CONFLICT ({pk}) DO NOTHING"

        with conn.cursor() as cur:
            for start in range(0, len(pdf), BATCH_SIZE):
                batch = pdf.iloc[start:start + BATCH_SIZE]
                rows = [tuple(None if (v is None or (isinstance(v, float) and str(v) == 'nan')) else v for v in row)
                        for _, row in batch.iterrows()]
                cur.executemany(sql, rows)
        conn.commit()
        print(f"  app.{tbl}: {len(pdf)} rows synced ✓")

print("\n✅ Production sync complete — all 4 tables in app.* schema")

In [0]:
# These tables are app-writable — NOT synced from Delta.
# Create on dev branch first, then replicate DDL to production.

DEV_ENDPOINT = f"projects/{PROJECT_ID}/branches/dev/endpoints/primary"
dev_ep = w.postgres.get_endpoint(name=DEV_ENDPOINT)
dev_host = dev_ep.status.hosts.host
dev_user = w.current_user.me().user_name
dev_token = w.postgres.generate_database_credential(endpoint=DEV_ENDPOINT).token

DDL_STATEMENTS = [
    # --- Chat state tables ---
    """CREATE TABLE IF NOT EXISTS app.conversations (
        id TEXT PRIMARY KEY,
        user_email TEXT NOT NULL,
        title TEXT,
        kind TEXT DEFAULT 'default',
        created_at TIMESTAMPTZ DEFAULT now(),
        updated_at TIMESTAMPTZ DEFAULT now()
    )""",
    """CREATE TABLE IF NOT EXISTS app.messages (
        id TEXT PRIMARY KEY DEFAULT gen_random_uuid()::text,
        conversation_id TEXT NOT NULL REFERENCES app.conversations(id) ON DELETE CASCADE,
        role TEXT NOT NULL,
        content TEXT,
        position INTEGER NOT NULL,
        trace_id TEXT,
        thinking JSONB,
        error TEXT,
        created_at TIMESTAMPTZ DEFAULT now()
    )""",
    """CREATE TABLE IF NOT EXISTS app.feedback (
        id TEXT PRIMARY KEY DEFAULT gen_random_uuid()::text,
        message_id TEXT NOT NULL,
        value TEXT NOT NULL CHECK (value IN ('up', 'down')),
        rationale TEXT,
        trace_id TEXT,
        mlflow_assessment_id TEXT,
        created_at TIMESTAMPTZ DEFAULT now()
    )""",
    # --- Writable operational table ---
    """CREATE TABLE IF NOT EXISTS app.work_orders_app (
        id TEXT PRIMARY KEY DEFAULT gen_random_uuid()::text,
        line_id TEXT NOT NULL,
        action_type TEXT NOT NULL CHECK (action_type IN ('pull_now', 'run_to_shift_end', 'expedite_parts_and_run')),
        part_id TEXT,
        drafted_wo TEXT,
        predicted_downtime_cost_avoided_usd DOUBLE PRECISION,
        status TEXT NOT NULL DEFAULT 'proposed' CHECK (status IN ('proposed', 'approved', 'executed', 'overridden')),
        approved_by TEXT,
        audit_trail JSONB DEFAULT '[]'::jsonb,
        created_at TIMESTAMPTZ DEFAULT now(),
        decided_at TIMESTAMPTZ
    )""",
]

with psycopg.connect(host=dev_host, dbname=DATABASE, user=dev_user,
                     password=dev_token, sslmode="require") as conn:
    with conn.cursor() as cur:
        for ddl in DDL_STATEMENTS:
            table_name = ddl.split("app.")[1].split()[0]
            cur.execute(ddl)
            print(f"  ✓ app.{table_name}")
    conn.commit()

print("\n✅ Chat state + work_orders_app tables created on dev branch")

In [0]:
# Create the same writable tables on production branch.
# Synced tables handle the read-only mirrors; these are the app-owned tables.

PROD_ENDPOINT = f"projects/{PROJECT_ID}/branches/production/endpoints/primary"
prod_ep = w.postgres.get_endpoint(name=PROD_ENDPOINT)
prod_host = prod_ep.status.hosts.host
prod_user = w.current_user.me().user_name
prod_token = w.postgres.generate_database_credential(endpoint=PROD_ENDPOINT).token

# Ensure app schema exists on production
with psycopg.connect(host=prod_host, dbname=DATABASE, user=prod_user,
                     password=prod_token, sslmode="require") as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS app")
        for ddl in DDL_STATEMENTS:
            table_name = ddl.split("app.")[1].split()[0]
            cur.execute(ddl)
            print(f"  ✓ app.{table_name} (production)")
    conn.commit()

print("\n✅ All app tables replicated to production branch")
print("\nProduction branch now has:")
print("  • Synced tables (managed): line_status, open_atrisk, maintenance_recommendations, parts")
print("  • Writable tables: conversations, messages, feedback, work_orders_app")

In [0]:
from databricks.sdk.service.postgres import Role, RoleRoleSpec, RoleIdentityType
import psycopg

SP_APP_ID  = "e2da75db-dd33-4ee9-abce-bd630d947140"
USER_EMAIL = "dinesh.kamalakkannan@databricks.com"

IDENTITIES = [
    (SP_APP_ID,   RoleIdentityType.SERVICE_PRINCIPAL, "sp-volta-app"),
    (USER_EMAIL,  RoleIdentityType.USER,              "user-dinesh"),
]

branches = {
    "dev":        f"projects/{PROJECT_ID}/branches/dev",
    "production": f"projects/{PROJECT_ID}/branches/production",
}

for branch_label, branch_path in branches.items():
    print(f"\n=== {branch_label} branch ===")

    # 1. Create Lakebase roles
    for identity, id_type, role_id in IDENTITIES:
        try:
            w.postgres.create_role(
                parent=branch_path,
                role=Role(spec=RoleRoleSpec(
                    postgres_role=identity,
                    identity_type=id_type,
                )),
                role_id=role_id,
            ).wait()
            print(f"  ✓ Created role: {identity} ({id_type.value})")
        except Exception as e:
            if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
                print(f"  ✓ Role already exists: {identity}")
            else:
                print(f"  ⚠ Role error for {identity}: {e}")

    # 2. Grant permissions (autocommit to prevent cascading aborts)
    endpoint_name = f"{branch_path}/endpoints/primary"
    ep = w.postgres.get_endpoint(name=endpoint_name)
    host = ep.status.hosts.host
    me = w.current_user.me().user_name
    token = w.postgres.generate_database_credential(endpoint=endpoint_name).token

    with psycopg.connect(host=host, dbname=DATABASE, user=me,
                         password=token, sslmode="require", autocommit=True) as conn:
        with conn.cursor() as cur:
            for identity, _, _ in IDENTITIES:
                pg_role = f'"{identity}"'
                grants = [
                    f'GRANT CONNECT ON DATABASE {DATABASE} TO {pg_role}',
                    f'GRANT USAGE ON SCHEMA app TO {pg_role}',
                    f'GRANT USAGE ON SCHEMA public TO {pg_role}',
                    f'GRANT SELECT ON ALL TABLES IN SCHEMA app TO {pg_role}',
                    f'GRANT SELECT ON ALL TABLES IN SCHEMA public TO {pg_role}',
                    f'GRANT INSERT, UPDATE, DELETE ON app.work_orders_app TO {pg_role}',
                    f'GRANT INSERT, UPDATE, DELETE ON app.conversations TO {pg_role}',
                    f'GRANT INSERT, UPDATE, DELETE ON app.messages TO {pg_role}',
                    f'GRANT INSERT, UPDATE, DELETE ON app.feedback TO {pg_role}',
                    f'ALTER DEFAULT PRIVILEGES IN SCHEMA app GRANT SELECT ON TABLES TO {pg_role}',
                    f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT SELECT ON TABLES TO {pg_role}',
                ]
                print(f"\n  Granting to {identity}:")
                for sql in grants:
                    try:
                        cur.execute(sql)
                        print(f"    ✓ {sql[:80]}")
                    except Exception as e:
                        print(f"    ⚠ {sql[:60]}... -> {e}")

print("\n✅ Permissions granted on both branches")

In [0]:
# Lakebase Search: hybrid BM25 full-text + vector index on app.parts
# Powers the agent's search_parts tool for the expedite-play lookup.
#
# PREREQUISITE — Enable Lakebase Search (one-time, irreversible, Beta):
#   1. Ensure Beta access — contact your Databricks account rep if not yet approved.
#   2. In the Databricks UI, open the Lakebase project (volta-ind).
#   3. Click "Settings" in the left navigation.
#   4. Under "Lakebase Search", click "Enable Lakebase Search".
#   5. WARNING: this restarts all computes (drops active connections) and cannot be undone.
#   6. Once enabled, the extensions become installable. Re-run this cell.
#
#   Without this step, CREATE EXTENSION will fail with:
#     "lakebase_text must be loaded via shared_preload_libraries"

import psycopg
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

PROJECT_ID = "volta-ind"
DATABASE = "databricks_postgres"

EXTENSION_DDL = [
    # pgvector MUST be installed first (lakebase_vector depends on it)
    "CREATE EXTENSION IF NOT EXISTS vector",
    "CREATE EXTENSION IF NOT EXISTS lakebase_vector CASCADE",
    "CREATE EXTENSION IF NOT EXISTS lakebase_text",
]

SEARCH_DDL = [
    # Add tsvector column combining part_name + description
    """ALTER TABLE app.parts
       ADD COLUMN IF NOT EXISTS search_tsv TSVECTOR""",
    # Populate the tsvector from existing data
    """UPDATE app.parts
       SET search_tsv = to_tsvector('english',
           coalesce(part_name, '') || ' ' || coalesce(description, ''))
       WHERE search_tsv IS NULL""",
    # Create BM25 index (lakebase_text) — corpus stats computed at build time
    """CREATE INDEX IF NOT EXISTS parts_search_bm25
       ON app.parts USING lakebase_bm25 (search_tsv)""",
]

for branch_id in ["production", "dev"]:
    endpoint = f"projects/{PROJECT_ID}/branches/{branch_id}/endpoints/primary"
    ep = w.postgres.get_endpoint(name=endpoint)
    host = ep.status.hosts.host
    user = w.current_user.me().user_name
    token = w.postgres.generate_database_credential(endpoint=endpoint).token

    print(f"\n=== {branch_id} ===")
    with psycopg.connect(host=host, dbname=DATABASE, user=user,
                         password=token, sslmode="require", autocommit=True) as conn:
        with conn.cursor() as cur:
            # Install extensions
            for ddl in EXTENSION_DDL:
                ext = ddl.split("EXISTS ")[1].split()[0]
                try:
                    cur.execute(ddl)
                    print(f"  \u2713 Extension {ext} installed")
                except Exception as e:
                    if "shared_preload" in str(e).lower():
                        print(f"  \u26a0 {ext}: Enable Lakebase Search in project settings first")
                        break
                    else:
                        print(f"  \u26a0 {ext}: {e}")
            else:
                # Extensions installed — proceed with search DDL
                for ddl in SEARCH_DDL:
                    action = ddl.strip().split("\n")[0][:60]
                    try:
                        cur.execute(ddl)
                        rows = getattr(cur, 'rowcount', 0)
                        print(f"  \u2713 {action}... ({rows} rows affected)")
                    except Exception as e:
                        print(f"  \u26a0 {action}: {e}")

    # Verify
    try:
        with psycopg.connect(host=host, dbname=DATABASE, user=user,
                             password=token, sslmode="require") as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT count(*) FROM app.parts WHERE search_tsv IS NOT NULL")
                cnt = cur.fetchone()[0]
                cur.execute("""SELECT indexname FROM pg_indexes
                               WHERE tablename = 'parts' AND indexname = 'parts_search_bm25'""")
                idx = cur.fetchone()
                print(f"  Indexed rows: {cnt}, BM25 index: {'\u2713' if idx else '\u2717'}")
    except Exception:
        pass

print("\n\u2705 Lakebase Search setup complete")
print("  BM25 query example:")
print("    SELECT part_id, part_name,")
print("      search_tsv <@> to_bm25query(to_tsvector('english', 'bearing vibration'), 'parts_search_bm25') AS score")
print("    FROM app.parts ORDER BY score LIMIT 5;")

In [0]:
# Reverse sync: Lakebase app.work_orders_app → Delta Lake SCD-2 table
# Tracks full history of work order status changes (proposed → approved → executed → overridden)
# Run periodically or on-demand to capture changes.

import psycopg
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

CATALOG = "tech_summit_jinie_group_catalog"
SCHEMA = "default"
SCD2_TABLE = f"{CATALOG}.{SCHEMA}.work_orders_app_scd2"
PROJECT_ID = "volta-ind"
DATABASE = "databricks_postgres"
PROD_ENDPOINT = f"projects/{PROJECT_ID}/branches/production/endpoints/primary"

# ── 1. Create SCD-2 target table in Delta (if not exists) ──
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCD2_TABLE} (
    id STRING,
    line_id STRING,
    action_type STRING,
    part_id STRING,
    drafted_wo STRING,
    predicted_downtime_cost_avoided_usd DOUBLE,
    status STRING,
    approved_by STRING,
    audit_trail STRING,
    created_at TIMESTAMP,
    decided_at TIMESTAMP,
    -- SCD-2 tracking columns
    effective_from TIMESTAMP NOT NULL,
    effective_to TIMESTAMP,
    is_current BOOLEAN NOT NULL
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"✓ SCD-2 table ready: {SCD2_TABLE}")

# ── 2. Read current state from Postgres ──
ep = w.postgres.get_endpoint(name=PROD_ENDPOINT)
prod_host = ep.status.hosts.host
prod_user = w.current_user.me().user_name
prod_token = w.postgres.generate_database_credential(endpoint=PROD_ENDPOINT).token

with psycopg.connect(host=prod_host, dbname=DATABASE, user=prod_user,
                     password=prod_token, sslmode="require") as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, line_id, action_type, part_id, drafted_wo,
                   predicted_downtime_cost_avoided_usd, status, approved_by,
                   audit_trail::text AS audit_trail, created_at, decided_at
            FROM app.work_orders_app
        """)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

if not rows:
    print("ℹ No work orders in Postgres yet — SCD-2 table created and ready for future syncs")
else:
    import pandas as pd
    pdf = pd.DataFrame(rows, columns=columns)
    staging_df = spark.createDataFrame(pdf)
    staging_df.createOrReplaceTempView("wo_staging")
    print(f"  Staging: {len(pdf)} rows from Postgres")

    # ── 3. SCD-2 MERGE ──
    # Step A: Close records whose mutable columns have changed
    closed = spark.sql(f"""
    MERGE INTO {SCD2_TABLE} t
    USING wo_staging s
    ON t.id = s.id AND t.is_current = true
    WHEN MATCHED AND (
        t.status       IS DISTINCT FROM s.status
        OR t.approved_by IS DISTINCT FROM s.approved_by
        OR t.decided_at  IS DISTINCT FROM s.decided_at
        OR t.drafted_wo  IS DISTINCT FROM s.drafted_wo
        OR t.audit_trail IS DISTINCT FROM s.audit_trail
    )
    THEN UPDATE SET
        effective_to = current_timestamp(),
        is_current   = false
    """)

    # Step B: Insert new current versions (new records + changed records whose old version was just closed)
    inserted = spark.sql(f"""
    INSERT INTO {SCD2_TABLE}
    SELECT
        s.id, s.line_id, s.action_type, s.part_id, s.drafted_wo,
        s.predicted_downtime_cost_avoided_usd, s.status, s.approved_by,
        s.audit_trail, s.created_at, s.decided_at,
        current_timestamp() AS effective_from,
        CAST(NULL AS TIMESTAMP) AS effective_to,
        true AS is_current
    FROM wo_staging s
    WHERE NOT EXISTS (
        SELECT 1 FROM {SCD2_TABLE} t
        WHERE t.id = s.id AND t.is_current = true
    )
    """)

    # ── 4. Summary ──
    stats = spark.sql(f"""
    SELECT
        count(*) AS total_rows,
        sum(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_records,
        sum(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS historical_records
    FROM {SCD2_TABLE}
    """).collect()[0]
    print(f"\n✅ SCD-2 reverse sync complete")
    print(f"  Total rows:  {stats.total_rows}")
    print(f"  Current:     {stats.current_records}")
    print(f"  Historical:  {stats.historical_records}")
    print(f"  Table:       {SCD2_TABLE}")